# Session 16 — Frontier & Synthesis

**Author:** PPSP lab  **·**  **Date:** 2026-05-28  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching · closing

## Purpose

The final session. We:

1. **Look back** on fifteen sessions in one sentence each, and watch the architecture come into focus.
2. **Look at the limits.** The De Palma et al. (2023) variational-quantum-algorithm
   limitation theorem closes the loop with the kind of broken "quantum-advantage" demos
   that motivated the whole course — and uses quantum OT itself as the analytic
   instrument.
3. **Look at the horizon.** The QOT taxonomy beyond the coupling SDP we built; the
   open problems (quantum transfer entropy, hyperscanning QOT, complexity of operator
   Sinkhorn at large scale).
4. **Run the course's greatest hits** in a single executable cell — every promised
   numerical result of M1–M4, replayed.
5. **Read the completed dictionary** and the bibliography for further study.

This session is mostly narrative. The mathematics is done; the synthesis remains.

## 0. What you'll achieve in this final session

- Recite the 15 sessions' main result in one sentence each.
- State the De Palma et al. (2023) VQE limitation theorem and explain why quantum OT is the right tool to prove it.
- List the **five** known QOT formulations beyond the coupling SDP and place each in context.
- Identify three concrete open research problems (quantum TE, hyperscanning QOT, large-scale operator Sinkhorn).
- Re-run every canonical numerical result of the course in a single `course_greatest_hits()` call.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum_ot.synthesis import course_greatest_hits

np.random.seed(42)
viz.use_course_style()

## 1. The fifteen sessions in one sentence each

| Session | Movement | One-sentence summary |
|---|---|---|
| S1 | (kickoff) | Optimal transport is moving a pile of mass at least cost; one POT call already does it. |
| S2 | M1 (quantum foundations) | A qubit is a complex superposition $\alpha|0\rangle + \beta|1\rangle$; Born rule gives the measurement statistics. |
| S3 | M1 | The density matrix $\rho$ unifies pure and mixed states; spectrum is intrinsic, basis isn't. |
| S4 | M1 | Tensor product builds composite systems; partial trace is the quantum marginal; entanglement breaks the classical joint-distribution mould. |
| S5 | M2 (information & geometry) | Entropy / KL / mutual information / transfer entropy — the classical information toolbox. |
| S6 | M2 | The simplex is curved: Fisher–Rao geometry on one hand, **mixture vs Wasserstein interpolation** the second geometry on the same simplex. |
| S7 | M2 | Quantum mutual information can hit 2 bits (Bell pair); quantum conditional entropy can go **negative**; Bures = quantum Fisher. |
| S8 | M3 (classical OT) | Monge ⊂ Kantorovich (mass-splitting is impossible for $T$ but trivial for $P$); Birkhoff–von Neumann pins down the equal-mass polytope. |
| S9 | M3 | $W_p$ is a metric; in 1-D it has a quantile closed form (sort and match); $W_1$ = area between CDFs. |
| S10 | M3 | Sinkhorn = 5-line matrix scaling = **KL projection of the Gibbs kernel** onto the transportation polytope (Amari's bridge). |
| S11 | M3 | Bures–Wasserstein on Gaussians = **Bures distance on density matrices** (S7) — the bridge to M4. |
| S12 | M4 (quantum OT) | Classical OT on diagonals is blind to coherences; non-commuting states need an intrinsic, operator-level notion. |
| S13 | M4 | Quantum coupling SDP: replace $P \ge 0$ by $\rho_{AB} \succeq 0$ with partial-trace marginals; cvxpy solves it in 5 lines. |
| S14 | M4 | Quantum Sinkhorn = **Umegaki projection of the matrix-exponential Gibbs kernel** $\exp(-C/\varepsilon)$ — Amari's bridge survives the lift. |
| S15 | M4 (capstone) | Synthetic Kuramoto dyad: QMI / Bures-coupling track injected $K$ alongside PLV; whether they *beat* PLV is open. |

The architecture: **M1 builds the language**, **M2 builds the geometry**, **M3 builds
the transport**, **M4 lifts everything to operators**. Each movement supplies a building
block for the next; the final dictionary connects them all.

## 2. The De Palma et al. (2023) VQE limitation theorem

**Setting.** A *variational quantum algorithm* (VQA) tries to prepare a target state
$\rho^\star$ (typically the ground state of some Hamiltonian $H$) by tuning a
parametrised circuit $|\psi(\theta)\rangle$ and minimising $\langle \psi(\theta)|H|\psi(\theta)\rangle$.

**The "broken demo" pattern.** A common claim: "this VQA finds the optimum exponentially
faster than any classical method". Researchers run a small example, see fast convergence,
and over-extrapolate.

**De Palma, Marvian, Rouzé, França, Wilde, Lloyd (2023).** Show that for a *large class
of cost Hamiltonians* (specifically, those with a Lipschitz structure under the
quantum-$W_1$ distance), **no shallow-depth VQA can achieve below a constant-multiplicative-error
ground-state energy**. The proof uses the quantum-$W_1$ Lipschitz norm to bound the
landscape: any state close (in $W_1$) to a product state inherits an energy bound from
the Hamiltonian's Lipschitz constant.

**Why this matters here.** Quantum OT is the analytic instrument that *proves* a no-go
result about VQA optimisation — the same machinery we built in M4 to *enable* coupling
measures *also* certifies that certain naive "quantum advantage" demos cannot work. The
course's principal tool has a constructive *and* a restrictive use.

This is the deepest closing message of the course: **building a powerful framework
includes building it sharp enough to recognise its own boundary**.

## 3. The QOT taxonomy — broader than the coupling SDP

Our principal definition (S13) is the **coupling SDP**: a bipartite density matrix
$\rho_{AB}$ with prescribed partial-trace marginals, minimising $\mathrm{tr}(C\rho_{AB})$.
The literature has at least four other genuinely-distinct definitions:

| Name | Reference | Idea | Property |
|---|---|---|---|
| **Coupling SDP** | De Palma–Trevisan (2021); Cole et al. (2023) | $\rho_{AB} \succeq 0$ with marginals; minimise $\mathrm{tr}(C\rho_{AB})$. **S13** | direct LP analogue; PSD-cone constraint |
| **Quantum Sinkhorn / entropic** | Peyré–Cuturi (2019); Pelikh–Gerolin | Same SDP with von-Neumann-entropy regulariser; matrix-exp Gibbs kernel. **S14** | scalable; Amari bridge to Umegaki KL |
| **Dynamic / Carlen–Maas** | Carlen–Maas (2014) | Replace continuity equation by a Lindblad-type generator; define $W_2$ via path length. | Riemannian; ties to quantum gradient flow |
| **Channel-based** | De Palma–Trevisan (2021) (other version) | Use Stinespring dilations of CPTP channels rather than couplings; minimise $\mathbb{E}\|X - Y\|^2$ over channel outputs. | no explicit coupling object; closer to communication theory |
| **Qubit-$W_1$ / Lipschitz** | De Palma, Marvian, Trevisan, Lloyd (2021) | Define a Lipschitz norm on multi-qubit observables and take the dual; specialised to many-body systems. | the analytic tool of the De Palma et al. (2023) result; *not* generally equal to coupling-SDP value |

**These are not all equivalent.** They reduce to each other on Gaussian states or on
classical (diagonal) states, but disagree off those special cases. The "right" choice
of quantum OT depends on the application — a fact that the course's syllabus (Trevisan,
2022) explicitly names as Trevisan's taxonomy.

## 4. Open problems

Three concrete questions for further study (and three places to start research):

1. **Quantum transfer entropy.** Classical transfer entropy (S5) measures directed
   information flow. No quantum analogue is universally accepted. Candidates exist
   (Lloyd-style channel TE, conditional QMI under projective measurements, Wigner-flow
   TE) but none satisfies all desiderata simultaneously. **A clean definition of
   directional information flow between quantum subsystems is open.**
2. **Hyperscanning QOT.** When two EEG-equipped human brains are simultaneously
   recorded ("hyperscanning"), the natural object is a *bipartite* time-frequency
   distribution. Does QOT-based coupling beat PLV / coherence on real EEG data? The S15
   synthetic Kuramoto study is a sanity check; a real-data study is the open question.
3. **Algorithmic complexity at scale.** The coupling SDP scales as
   $\mathcal{O}((d_A d_B)^{O(1)})$ in the dimension — fine for qubits, painful for
   ten qubits, impossible for many. Quantum Sinkhorn improves scaling but
   matrix-exponential operations are still polynomial in $d_A d_B$.
   **An efficient quantum-OT algorithm at large many-body scales is open.**

These are not the *only* open problems — but they are the ones this course's machinery
is best placed to attack.

## 5. The course's greatest hits — every key result in one cell

The `course_greatest_hits()` function in `qot_course.quantum_ot.synthesis` re-runs the
canonical numerical demonstrations from S5 through S14 in a single call. Every entry
corresponds to a numerical promise made in one session.

In [ ]:
hits = course_greatest_hits()
print(f"{'result':<55s} {'value':>14s}")
print("-" * 72)
for label, value in hits.items():
    print(f"{label:<55s} {value:>14.6f}")

**Read the table.** Every line is a promise delivered:

- **S5: H(fair coin) = 1.000 bit.** Classical entropy as average surprise.
- **S5: I(independent) = 0**, **I(correlated) = 1 bit**. Mutual information's two endpoints.
- **S6: mixture midpoint mass-in-middle ≈ 0**, **W₂ midpoint mass-in-middle ≈ 1**. The two-geometries-on-one-simplex punchline.
- **S7: Bell QMI = 2 bits**, **Bell S(A|B) = −1 bit**. Doubled MI and negative conditional entropy — both impossible classically.
- **S7: S(|+⟩⟨+| ∥ I/2) = 1 bit**, **d_B(|+⟩⟨+|, I/2) > 0**. Same Z-diagonal, different states.
- **S9: W₂ closed form == W₂ via LP** to machine precision. The 1-D quantile gift.
- **S11: √(BW matrix term on density matrices) == d_B from S7.** The bridge between Bures–Wasserstein on Gaussians and Bures on density matrices.
- **S13: QOT(|+⟩⟨+|, I/2) > 0**, **SWAP-QOT²(|0⟩, |1⟩) = 1**. The SDP delivering on M4's promise.
- **S14: ε·S_Umegaki(P∥K) == tr(C·P) − ε·S(P)** to ~10⁻⁴. **Amari's quantum bridge.**

If any single one of these were *wrong*, the corresponding session would need revision.
They all hold simultaneously — the course's architectural consistency is intact.

## 6. The completed dictionary — final form

The classical ↔ quantum dictionary the course has built, in one place:

| Classical | Quantum | First seen |
|-----------|---------|-----------|
| probability vector $a \in \Delta^n$ | density matrix $\rho \in \mathcal{S}(\mathcal{H})$ | S3 |
| event probability $p(x)$ | Born rule $\|\langle x|\psi\rangle\|^2$ | S2 |
| marginal | partial trace | S4 |
| product distribution $p_X p_Y$ | product state $\rho_A \otimes \rho_B$ | S4 |
| Markov kernel | quantum channel (CPTP map) | S4 |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ | S3, S5, S7 |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ | S7 |
| mutual information $I(X;Y) \le \min(H_X, H_Y)$ | quantum MI $I(A{:}B)$ — *up to twice the classical bound* | S7 |
| conditional entropy $H(X|Y) \ge 0$ | quantum CMI $S(A|B)$ — **can be negative** | S7 |
| Fisher–Rao metric | Bures / quantum Fisher information | S6, S7 |
| transfer entropy $\mathrm{TE}$ | (open — no universally accepted definition) | S5, S15+ |
| transport plan / coupling $P \in T(a, b)$ | **quantum coupling $\rho_{AB} \succeq 0$ with marginals** | S13 |
| Kantorovich LP $\min \langle C, P\rangle$ | **quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$** | S13 |
| transportation polytope $T(a, b)$ | **spectrahedron of quantum couplings** | S13 |
| Birkhoff polytope / Birkhoff–von Neumann | (analogous structural result open) | S8, S16 |
| Wasserstein-$p$ distance $W_p$ | **quantum Wasserstein** (multiple non-equivalent definitions) | S13–S14, S16 |
| 1-D quantile closed form $W_p^p = \int |F^{-1} - G^{-1}|^p$ | (no analogue — operators don't sort) | S9 |
| $W_1 = \int |F_\mu - F_\nu|\,\mathrm{d}x$ | (qubit-$W_1$: Lipschitz dual, no integral form) | S9, S16 |
| Gaussian Bures–Wasserstein $W_2$ | Bures distance on density matrices (bridge!) | **S11** |
| McCann displacement geodesic | (quantum displacement / Carlen–Maas dynamic) | S14, S16 |
| Sinkhorn algorithm | **quantum Sinkhorn** (matrix exp + partial trace) | S14 |
| Sinkhorn = KL projection of $K = e^{-C/\varepsilon}$ | **quantum Sinkhorn = Umegaki projection of $K = \exp(-C/\varepsilon)$** | S14 |
| Benamou–Brenier / Otto Riemannian metric | Carlen–Maas quantum Riemannian metric | S11, S16 |

This dictionary is the *spine of the course*. Every row was earned with code, a
derivation, and a "Read the figure" explanation. The empty entries (transfer entropy,
Birkhoff–von Neumann analogue, 1-D operator quantile) are honest acknowledgements of
where the analogy breaks down — and where research opens up.

## 7. Reading list for further study

The bibliography of the course, organised by topic:

**Quantum information & geometry (M1, M2)**
- M. A. Nielsen, I. L. Chuang, *Quantum Computation and Quantum Information* (Cambridge, 2010).
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge, 2017).
- S.-i. Amari, *Information Geometry and Its Applications* (Springer, 2016).
- N. J. Cerf, C. Adami, "Negative entropy and information in quantum mechanics", PRL **79**, 5194 (1997).

**Classical optimal transport (M3)**
- C. Villani, *Topics in Optimal Transportation* (AMS, 2003) — definitive textbook.
- G. Peyré, M. Cuturi, *Computational Optimal Transport*, Found. Trends ML (2019) — the practical reference.
- M. Cuturi, "Sinkhorn distances", NeurIPS (2013) — the algorithm that made OT practical.
- F. Otto, "The geometry of dissipative evolution equations", Comm. PDE **26**, 101 (2001).

**Quantum optimal transport (M4)**
- D. Trevisan, *Optimal transport methods for quantum systems* (arXiv:2202.02091, 2022) — survey of the field.
- G. De Palma, D. Trevisan, "Quantum optimal transport with quantum channels", Ann. Henri Poincaré **22**, 3199 (2021).
- G. De Palma, M. Marvian, D. Trevisan, S. Lloyd, "The quantum Wasserstein distance of order 1", IEEE Trans. Inf. Theory **67**, 6627 (2021).
- G. Peyré, L. Chizat, F.-X. Vialard, B. Schmitzer, "Quantum entropic regularization of matrix-valued optimal transport", European J. Appl. Math. **30**, 1079 (2019).
- E. Carlen, J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability", Comm. Math. Phys. **331**, 887 (2014).
- G. De Palma, M. Marvian, C. Rouzé, D. S. França, M. M. Wilde, S. Lloyd, "Limitations of variational quantum algorithms: a quantum optimal transport approach" (arXiv:2204.03455, 2022/2023).

**Hyperscanning / brain coupling (for the capstone direction)**
- J.-P. Lachaux, E. Rodriguez, J. Martinerie, F. Varela, "Measuring phase synchrony in brain signals", Hum. Brain Mapp. **8**, 194 (1999).
- M. Treves, S. Panzeri, "The upward bias in measures of information derived from limited data samples", Neural Comput. **7**, 399 (1995).
- F. Babiloni, L. Astolfi, "Social neuroscience and hyperscanning techniques", Neurosci. Biobehav. Rev. **44**, 76 (2014).

The course's own materials remain at `notebooks/`, `summaries/`, and
`src/qot_course/` — re-runnable, modifiable, and (most importantly) *honest about
their own limits*.

## 8. Conclusions — an honest accounting

What this course built:

- A **complete classical ↔ quantum dictionary** spanning information theory,
  information geometry, classical OT, and quantum OT.
- A **single computational framework** (cvxpy + numpy + scipy + POT) capable of
  solving the quantum OT SDP, its entropic regularisation, and the bridge identities to
  Bures / Umegaki.
- **Concrete, executable demonstrations** of every theoretical promise: from the
  $|+\rangle\langle+|$ vs $I/2$ punchline through the Amari bridge identity to a
  synthetic Kuramoto coupling sweep.
- A **synthesis function** (this session) that re-runs every canonical result, serving
  as both pedagogical demo and regression test.

What this course did **not** prove:

- That quantum OT measures beat classical baselines (PLV, correlation) on real-world
  data. The S15 sanity check leaves the question open.
- That the coupling SDP is the *right* quantum OT formulation for any specific
  application. Five non-equivalent definitions exist; choosing among them is an
  application-level decision.
- That naive VQE-style "quantum advantage" claims are valid. De Palma et al. (2023)
  explicitly *forbid* a class of such claims using the very machinery we built.

The course delivered **a framework, a vocabulary, and a discipline**. The framework is
quantum OT in its coupling-SDP form. The vocabulary is the 21-row classical ↔ quantum
dictionary. The discipline is: *prove what you claim, verify against known limits,
honor open problems*.

Where to go from here:

- **For students of mathematics**: Villani (2003), Amari (2016), Trevisan (2022).
- **For neuroscientists**: implement the S15 Kuramoto experiment on real EEG data;
  compare to PLV / coherence; publish whatever you find — *especially* if you find that
  the quantum machinery does not beat the baselines, because that result is more
  important than a positive one.
- **For everyone**: re-read the course's "Honest caveats" sections. They are the most
  valuable parts.

---

*Course end.* Thank you for the 30 hours.

**References:** all primary sources listed in §7. Previous: `notebooks/04_QuantumOptimalTransport/04_capstone.ipynb`.